# Simulating Atmospheric Turbulence

When light from distant stars passes through Earth's atmosphere, it encounters turbulent layers of air with varying temperatures and densities. These atmospheric disturbances act like distorting lenses, blurring astronomical images and limiting the resolution of ground-based telescopes.

Understanding and simulating atmospheric turbulence is essential for:
* **Adaptive Optics (AO)**: Designing systems that correct for atmospheric distortion in real-time
* **High-Contrast Imaging**: Predicting how turbulence affects the detection of faint exoplanets
* **Telescope Site Selection**: Evaluating atmospheric conditions at potential observatory locations
* **Instrument Design**: Testing how optical systems perform under realistic seeing conditions

In this tutorial, we'll explore how HCIPy models atmospheric turbulence using phase screens, from simple single-layer models to realistic multi-layer atmospheric profiles.



In [ ]:
from hcipy import *
import numpy as np
import matplotlib.pyplot as plt
from tqdm import tqdm

## Understanding Atmospheric Turbulence

Atmospheric turbulence is characterized by variations in the refractive index of air, caused by temperature and density fluctuations. In optical wave propagation, these variations introduce phase delays that distort the wavefront.

The strength of atmospheric turbulence is typically quantified by:

* **Fried parameter (r₀)**: The aperture diameter over which the root-mean-square (RMS) wavefront aberration is approximately 1 radian. Larger r₀ means better seeing.
* **Cn²**: The refractive index structure constant, which describes the strength of turbulence at different altitudes
* **Outer scale (L₀)**: The largest scale of turbulent eddies, typically 10-100 meters

Let's set up our simulation with a VLT-sized telescope (8.2 meters) observing at 1 micron wavelength.

In [ ]:
# Telescope parameters
pupil_diameter = 8.2  # meters (VLT-sized telescope)
wavelength = 1.5e-6  # meters (1.5 micron)

# Create computational grids
# Pupil grid: where the telescope aperture is defined
pupil_grid = make_pupil_grid(256, pupil_diameter * 1.2)

# Focal grid: where we observe the PSF
# q=4 means 4 pixels per λ/D, num_airy=32 means 32 λ/D radius field of view
spatial_resolution = wavelength / pupil_diameter
focal_grid = make_focal_grid(q=4, num_airy=32, spatial_resolution=wavelength / pupil_diameter)

# Create propagator to go from pupil to focal plane
prop = FraunhoferPropagator(pupil_grid, focal_grid)

# Create telescope aperture
aperture = make_vlt_aperture()(pupil_grid)

print(f"Telescope diameter: {pupil_diameter} m")
print(f"Observing wavelength: {wavelength*1e9:.0f} nm")
print(f"Pupil grid: {pupil_grid.dims} points")
print(f"Focal grid: {focal_grid.dims} points")

## Creating a Phase Screen

The key to simulating atmospheric turbulence is generating realistic phase screens that represent the wavefront distortion caused by atmospheric layers. HCIPy uses the method developed by Assemat et al. (2006) to create effectively infinite phase screens.

This technique generates a phase screen that can be 'scrolled' in time to simulate the wind-driven movement of atmospheric turbulence across the telescope aperture.

In [ ]:
# Atmospheric turbulence parameters
fried_parameter = 0.2  # meters (r0 = 20 cm - typical good seeing)
outer_scale = 20  # meters (L0)
velocity = 10  # meters/second (wind speed)

# Convert Fried parameter to Cn squared
# The Fried parameter depends on wavelength: r0 is proportional to lambda^(6/5)
# We specify the wavelength at which we want r0 (typically 500 nm)
Cn_squared = Cn_squared_from_fried_parameter(fried_parameter, wavelength=500e-9)

print("Atmospheric Parameters:")
print(f"  Fried parameter (r0): {fried_parameter} m at 500 nm")
print(f"  Outer scale (L0): {outer_scale} m")
print(f"  Wind velocity: {velocity} m/s")
print(f"  Cn squared: {Cn_squared:.2e} m^(-2/3)")

# Create the atmospheric layer
layer = InfiniteAtmosphericLayer(
    pupil_grid,
    Cn_squared=Cn_squared,
    L0=outer_scale,
    velocity=velocity,
    seed=0
)

print("\nAtmospheric phase screen created")

Now let's visualize the phase screen. The phase variations across the telescope aperture represent the optical path differences introduced by atmospheric turbulence.

In [ ]:
# Get the phase screen at our observing wavelength
phase = layer.phase_for(wavelength)

# Visualize the phase screen (masked to show only the telescope aperture)
plt.figure(figsize=(10, 8))
im = imshow_field(phase, cmap='RdBu_r', mask=aperture, vmin=-10, vmax=10)
plt.colorbar(im, label='Phase (radians)')
plt.title(f'Atmospheric Phase Screen\nr0 = {fried_parameter}m at {wavelength*1e9:.0f}nm')
plt.xlabel('x (m)')
plt.ylabel('y (m)')
plt.axis('equal')
plt.tight_layout()
plt.show()

# Calculate wavefront error statistics
phase_inside_aperture = phase[aperture > 0]
rms_wfe = np.sqrt(np.mean(phase_inside_aperture**2))
peak_to_valley = np.max(phase_inside_aperture) - np.min(phase_inside_aperture)

print(f"Wavefront Error Statistics (inside aperture):")
print(f"  RMS: {rms_wfe:.2f} rad ({rms_wfe * wavelength * 1e9:.1f} nm)")
print(f"  Peak-to-Valley: {peak_to_valley:.2f} rad ({peak_to_valley * wavelength * 1e9:.1f} nm)")

## Impact on Image Quality

The phase errors across the telescope aperture directly affect the Point Spread Function (PSF). Let's compare the diffraction-limited PSF (no atmosphere) with the turbulence-degraded PSF.

In [ ]:
# Create a plane wave entering the telescope
wf = Wavefront(aperture, wavelength)

# Apply atmospheric turbulence and propagate to focal plane
wf_atmos = layer(wf)
img_turbulent = prop(wf_atmos)

# For comparison, propagate without turbulence (diffraction-limited)
img_diffraction_limited = prop(wf)

# Visualize comparison
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Diffraction-limited PSF
im0 = imshow_field(
    np.log10(img_diffraction_limited.intensity / img_diffraction_limited.intensity.max()),
    vmin=-5, vmax=0, cmap='inferno', ax=axes[0]
)
axes[0].set_title('Diffraction-Limited (No Atmosphere)', fontsize=12)
axes[0].set_xlabel('x (lambda/D)')
axes[0].set_ylabel('y (lambda/D)')
plt.colorbar(im0, ax=axes[0], label='log10(Intensity)')

# Turbulent PSF
im1 = imshow_field(
    np.log10(img_turbulent.intensity / img_turbulent.intensity.max()),
    vmin=-5, vmax=0, cmap='inferno', grid_units=spatial_resolution, ax=axes[1]
)
axes[1].set_title(f'With Atmospheric Turbulence (r0 = {fried_parameter}m)', fontsize=12)
axes[1].set_xlabel('x (lambda/D)')
axes[1].set_ylabel('y (lambda/D)')
plt.colorbar(im1, ax=axes[1], label='log10(Intensity)')

plt.tight_layout()
plt.show()

# Calculate Strehl ratio (measure of image quality)
strehl_dl = np.max(img_diffraction_limited.intensity) / np.sum(img_diffraction_limited.intensity)
strehl_turb = np.max(img_turbulent.intensity) / np.sum(img_turbulent.intensity)
print(f"\nImage Quality Comparison:")
print(f"  Diffraction-limited peak intensity: {strehl_dl:.4f}")
print(f"  Turbulent peak intensity: {strehl_turb:.4f}")
print(f"  Strehl ratio: {strehl_turb/strehl_dl:.2%}")

## Time Evolution - Frozen Flow

Atmospheric turbulence is not static, it evolves as wind moves turbulent layers across the telescope aperture. This frozen flow hypothesis assumes that the turbulence pattern remains frozen while being transported by wind.

The InfiniteAtmosphericLayer simulates this by scrolling the phase screen. We can control the evolution using the t attribute (time in seconds) or by using the ``evolve_until`` method.

In [ ]:
# Create an animation showing the atmospheric evolution
duration = 1.0  # seconds
fps = 40
num_frames = int(duration * fps)
time_step = duration / num_frames

# Create the MP4 writer
writer = FFMpegWriter('atmosphere_evolution.mp4', framerate=10)

print(f"Creating animation with {num_frames} frames...")

for t in tqdm(np.arange(num_frames) * time_step):
    # Set the time (this scrolls the phase screen)
    layer.evolve_until(t)

    # Propagate through atmosphere at this time
    wf_atmos = layer(wf)
    img = prop(wf_atmos)

    # Get the phase screen at our wavelength for plotting purposes.
    phase = layer.phase_for(wavelength)

    # Create figure with phase and PSF side by side
    fig, axes = plt.subplots(1, 2, figsize=(14, 6))

    # Left: Phase pattern
    im0 = imshow_field(phase, cmap='RdBu_r', mask=aperture, vmin=-10, vmax=10, ax=axes[0])
    axes[0].set_title(f'Phase Screen (t = {t*1000:.0f} ms)', fontsize=12)
    axes[0].set_xlabel('x (m)')
    axes[0].set_ylabel('y (m)')
    plt.colorbar(im0, ax=axes[0], label='Phase (rad)')

    # Right: PSF
    im1 = imshow_field(np.log10(img.intensity / img.intensity.max()), vmin=-4, cmap='inferno', grid_units=spatial_resolution, ax=axes[1])
    axes[1].set_title('Point Spread Function', fontsize=12)
    axes[1].set_xlabel('x (lambda/D)')
    axes[1].set_ylabel('y (lambda/D)')
    plt.colorbar(im1, ax=axes[1], label='log10(Intensity)')

    # Add Strehl ratio text
    strehl = np.max(img.intensity) / np.sum(img.intensity)
    fig.suptitle(f'Strehl Ratio: {strehl/strehl_dl:.1%}', fontsize=14, y=0.98)

    plt.tight_layout(rect=[0, 0, 1, 0.96])

    # Add frame to animation
    writer.add_frame(fig)
    plt.close(fig)

# Close the writer to create the MP4
writer.close()

# Display the animation inline
writer

## Multi-Layer Atmospheric Models

Real atmospheric turbulence is distributed across multiple layers at different altitudes, each with different wind speeds and turbulence strengths. The Mauna Kea atmospheric model is a well-studied profile based on observations at one of the world's premier observatory sites.

Multi-layer models are essential for accurate simulations because:
* Different layers have different wind speeds and directions
* High-altitude layers cause anisoplanatism (different parts of the sky see different turbulence)
* The cumulative effect produces more realistic turbulence statistics

In [ ]:
# Create Mauna Kea atmospheric profile
# This returns a list of layers at different altitudes
layers = make_mauna_kea_atmospheric_layers(pupil_grid, outer_scale=outer_scale)

# Randomize layer velocity orientations
for layer in layers:
    angle = np.random.uniform(0, 2 * np.pi)
    velocity = np.linalg.norm(layer.velocity)

    layer.velocity = [np.cos(angle) * velocity, np.sin(angle) * velocity]

# Combine layers into a MultiLayerAtmosphere
# scintillation=True enables amplitude variations (important for high layers)
atmos = MultiLayerAtmosphere(layers, scintillation=True)

print(f"Mauna Kea Atmospheric Profile:")
print(f"  Number of layers: {len(layers)}")
print(f"\nLayer details:")
for i, layer in enumerate(layers):
    print(f"  Layer {i+1}: {layer.height:5.0f}m altitude, "
          f"Cn squared={layer.Cn_squared:.2e}, velocity={layer.velocity} m/s")

### Scintillation: Amplitude Effects

While phase screens capture the wavefront distortion, propagation through multiple turbulent layers also causes amplitude variations known as scintillation. This becomes particularly noticeable when light propagates over long distances through the atmosphere.

In [ ]:
# Set the turbulence strength
# r0 = 0.1m at 550nm corresponds to approximately 0.6 arcsec seeing
atmos.Cn_squared = Cn_squared_from_fried_parameter(0.1, wavelength=550e-9)
atmos.reset()  # Reset phase screens

# Create input wavefront
wf_in = Wavefront(pupil_grid.ones(), wavelength=wavelength)

# Create animation showing scintillation evolution
duration = 1  # seconds
fps = 40
num_frames = int(duration * fps)
time_step = duration / num_frames

writer = FFMpegWriter('scintillation_evolution.mp4', framerate=10)

print(f"Creating scintillation animation with {num_frames} frames...")

for t in tqdm(np.arange(num_frames) * time_step):
    # Evolve the atmosphere
    atmos.evolve_until(t)

    # Propagate through the multi-layer atmosphere
    wf_out = atmos(wf_in)

    # Create figure with intensity and phase side by side
    fig, axes = plt.subplots(1, 2, figsize=(14, 6))

    # Left: Intensity (showing scintillation)
    im0 = imshow_field(wf_out.intensity * aperture, cmap='gray', ax=axes[0], vmin=0, vmax=3)
    axes[0].set_title(f'Intensity (t = {t*1000:.0f} ms)', fontsize=12)
    axes[0].set_xlabel('x (m)')
    axes[0].set_ylabel('y (m)')
    plt.colorbar(im0, ax=axes[0], label='Intensity')

    # Right: Phase
    im1 = imshow_field(wf_out.phase * aperture, cmap='RdBu_r', ax=axes[1], vmin=-3, vmax=3)
    axes[1].set_title('Phase', fontsize=12)
    axes[1].set_xlabel('x (m)')
    axes[1].set_ylabel('y (m)')
    plt.colorbar(im1, ax=axes[1], label='Phase (rad)')

    # Calculate scintillation statistics
    intensity_in_aperture = wf_out.intensity[aperture > 0]
    scintillation_index = np.std(intensity_in_aperture) / np.mean(intensity_in_aperture)

    fig.suptitle(f'Scintillation Index: {scintillation_index:.3f}', fontsize=14, y=0.98)
    plt.tight_layout(rect=[0, 0, 1, 0.96])

    # Add frame to animation
    writer.add_frame(fig)
    plt.close(fig)

# Close the writer to create the MP4
writer.close()

# Display the animation inline
writer

## Varying Turbulence Strength

The atmospheric model allows on-the-fly changes to turbulence strength, enabling simulations of varying seeing conditions. Let's see how different Fried parameters affect image quality.

In [ ]:
# Reset atmosphere
atmos.reset()

# Test different seeing conditions
seeing_arcsec = [1.0, 0.5, 0.25]  # Approximate seeing in arcseconds

fig, axes = plt.subplots(1, 3, figsize=(16, 5))

for i, seeing in enumerate(seeing_arcsec):
    # Set turbulence strength
    r0 = seeing_to_fried_parameter(seeing)
    atmos.Cn_squared = Cn_squared_from_fried_parameter(r0)
    atmos.reset()

    # Propagate and get PSF
    wf_out = atmos(wf_in)
    img = prop(wf_out)

    # Display
    im = imshow_field(
        np.log10(img.intensity / img.intensity.max()),
        vmin=-3, cmap='inferno', grid_units=spatial_resolution, ax=axes[i]
    )
    axes[i].set_title(f'r0 = {r0*100:.0f} cm at 550nm\n({seeing:.2f} arcsec seeing)', fontsize=11)
    axes[i].set_xlabel('x (lambda/D)')
    axes[i].set_ylabel('y (lambda/D)')

    # Calculate metrics
    # Note: Calculate Strehl relative to DL PSF from multi-layer simulation
    strehl = np.max(img.intensity) / np.sum(img.intensity)
    axes[i].text(0.02, 0.98, f'Strehl: {strehl:.2%}',
                 transform=axes[i].transAxes, verticalalignment='top',
                 bbox=dict(boxstyle='round', facecolor='white', alpha=0.8))

plt.suptitle('Effect of Seeing on Image Quality', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

Key Observations:
- Better seeing (larger r0) produces tighter PSFs
- Poor seeing (small r0) causes significant image degradation
- Multi-layer atmosphere produces more realistic speckle patterns

## Summary and Best Practices

In this tutorial, we explored atmospheric turbulence simulation in HCIPy:

**Key Concepts:**
* **Phase Screens** represent wavefront distortion from atmospheric turbulence
* **Fried Parameter (r0)** characterizes turbulence strength - larger is better
* **Time Evolution** simulates wind-driven turbulence movement
* **Multi-Layer Models** provide realistic turbulence profiles for different altitudes
* **Scintillation** captures amplitude variations from extended propagation

In [ ]:
import os
os.remove('atmosphere_evolution.mp4')
os.remove('scintillation_evolution.mp4')